# Download general release data

Please keep the data in "data" folder localy. Don't commit large data to the repo!

In [ ]:
path_to_release_folder = "gs://open-targets-data-releases/25.06"

In [ ]:
!mkdir -p ../../data/25.06
!mkdir -p ../../data/25.06/output

In [ ]:
study_index_path = "../../data/25.06/output/study"
credible_set_path = "../../data/25.06/output/credible_set"
coloc_path = "../../data/25.06/output/colocalisation_coloc"
ecaviar_path = "../../data/25.06/output/colocalisation_ecaviar"
target_path = "../../data/25.06/output/target"
evidence_path = "../../data/25.06/output/evidence"
l2g_prediction_path = "../../data/25.06/output/l2g_prediction"
disease_path = "../../data/25.06/output/disease"
variant_path = "../../data/25.06/output/variant"
feature_matrix_path = "../../data/25.06/output/l2g_feature_matrix"
target_prioritisation_path = "../../data/25.06/output/target_prioritisation"
drug_molecule_path = "../../data/25.06/output/drug_molecule"

`gs://open-targets-data-releases` is a requester-pays bucket: set `BILLING_PROJECT` to a Google Cloud
project you can bill. The same release is free over HTTPS from
https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.06/ (`output/` and
`intermediate/l2g_feature_matrix`), if you prefer to fetch it into the paths above yourself.

In [ ]:
BILLING_PROJECT = "open-targets-genetics-dev"

In [ ]:
!gcloud auth login

In [ ]:
!gcloud storage --billing-project $BILLING_PROJECT ls "$path_to_release_folder/output/"

In [ ]:
def download_the_folder_from_gs_to_local(gs_folder_path: str, local_folder_path: str):
    !mkdir -p "$local_folder_path"
    !gcloud storage --billing-project=$BILLING_PROJECT rsync -r --delete-unmatched-destination-objects \
      "$path_to_release_folder/$gs_folder_path" "$local_folder_path"

In [ ]:
download_the_folder_from_gs_to_local("output/study", study_index_path)

In [ ]:
download_the_folder_from_gs_to_local("output/credible_set", credible_set_path)

In [ ]:
download_the_folder_from_gs_to_local("output/evidence", evidence_path)

In [ ]:
download_the_folder_from_gs_to_local("output/colocalisation_coloc", coloc_path)
download_the_folder_from_gs_to_local("output/colocalisation_ecaviar", ecaviar_path)

In [ ]:
download_the_folder_from_gs_to_local("output/target", target_path)
download_the_folder_from_gs_to_local("output/l2g_prediction", l2g_prediction_path)
download_the_folder_from_gs_to_local("output/disease", disease_path)
download_the_folder_from_gs_to_local("output/variant", variant_path)
download_the_folder_from_gs_to_local("output/target_prioritisation", target_prioritisation_path)
download_the_folder_from_gs_to_local("output/drug_molecule", drug_molecule_path)

In [ ]:
download_the_folder_from_gs_to_local("intermediate/l2g_feature_matrix", feature_matrix_path)

# Project inputs from Zenodo

Everything the pipeline reads that is not in the Open Targets release is deposited in Zenodo
(all versions: DOI [10.5281/zenodo.23736324](https://doi.org/10.5281/zenodo.23736324), CC0). Each archive holds
paths relative to the repository root and is extracted there; see the record's `README.md` for what
each dataset is.

| Archive | Size | Contents |
| ------- | ---- | -------- |
| `inputs.tar.gz` | 206 MB | L2G predictions, training set and held-out split, effector gene list, gene categories, combined evidence, target annotations, genetic correlations, enhancer–gene links (the rows the pipeline uses), gene therapeutic areas, per-gene effective-independent-trait counts |
| `intermediates.tar.gz` | 1.1 GB | `data/intermediate_files_refactor/`, the output of chapter 01: fetch it to skip chapter 01 and start at chapter 02 |
| `outputs.tar.gz` | 8 MB | `results/` and the supplementary table sheets, as published |

`ARCHIVES` below fetches the inputs only. Add `intermediates.tar.gz` to skip chapter 01.

In [ ]:
import hashlib
import json
import tarfile
import urllib.request
from pathlib import Path

# The all-versions record: the API redirects it to the latest published version.
ZENODO_RECORD = "23736324"
ARCHIVES = ["inputs.tar.gz"]
REPO_ROOT = Path("../..").resolve()

with urllib.request.urlopen(f"https://zenodo.org/api/records/{ZENODO_RECORD}") as response:
    record_files = {f["key"]: f for f in json.load(response)["files"]}


def md5(path: Path) -> str:
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_and_extract(name: str) -> None:
    """Download one archive from the record, check its MD5, and extract it at the repository root."""
    entry = record_files[name]
    target = REPO_ROOT / "data" / "zenodo" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    expected = entry["checksum"].removeprefix("md5:")
    if not target.exists() or md5(target) != expected:
        urllib.request.urlretrieve(entry["links"]["self"], target)
    observed = md5(target)
    assert observed == expected, f"{name}: MD5 {observed}, expected {expected}"
    with tarfile.open(target) as archive:
        archive.extractall(REPO_ROOT, filter="data")
    print(f"{name}: {entry['size'] / 1e6:,.0f} MB, MD5 ok, extracted")


for name in ARCHIVES:
    fetch_and_extract(name)

# Third-party file

The human-knockout gene set in `01-data-preparation/11_gene_sets.ipynb` comes from Supplementary
Table 8 of Sun et al. 2024, *A deep catalogue of protein-coding variation in 983,578 individuals*,
Nature, https://doi.org/10.1038/s41586-024-07556-0. The publisher blocks scripted downloads:
download `41586_2024_7556_MOESM8_ESM.csv` from the article page into `data/`.

In [ ]:
knockouts = REPO_ROOT / "data" / "41586_2024_7556_MOESM8_ESM.csv"
assert knockouts.exists(), f"download Supplementary Table 8 of Sun et al. 2024 to {knockouts}"
assert md5(knockouts) == "9ae619b423b59df5af56f6923e6e9bb3"
print("Sun et al. 2024 Supplementary Table 8: MD5 ok")

# Where the Zenodo inputs came from

For the record, the internal locations these were copied from (not accessible outside Open Targets):

| Dataset | Original location |
| ------- | ----------------- |
| `data/25.06/irene_1208_l2g_predictions` | `gs://ot-team/irene/l2g/xgboost/1208/l2g_predictions` |
| `data/l2g_training_set/` | the original analysis's L2G training-set archive (internal) |
| `data/intermediate_files/EGL.parquet` | `gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/20250625_EGL_2506_0.95_otg_chembl.parquet` |
| `data/intermediate_files/list_of_genes_32_categories` | `gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_ofgenes_32_categories.parquet` |
| `data/intermediate_files/combined_evidence_with_measurements` | `gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/combined_evidence_with_measurements.parquet` |
| `data/intermediate_files/annotated_targets_wide.parquet` | `gs://ot-team/irene/cautionary_targets/annotated_targets_wide.parquet` |
| `data/intermediate_files/regulatoryToGene` | `gs://genetics-portal-dev-analysis/xg1/epiraction/processed_epiraction_v1.6.parquet`, filtered to score >= 0.05 and overlap with a replicated lead variant |
| `data/intermediate_files/canonical_pairwise_table` | `gs://genetics-portal-dev-analysis/xg1/LDSC_pyspark/outputs/release_25_06/canonical_pairwise_table` |

The Pharmaprojects inputs of `01-data-preparation/12_drug_pair_tables.ipynb`
(`minikel_etal_processed_data_v2.csv`) are not deposited; that notebook skips them when absent.

# Static assets

Five files are shipped as-is rather than generated: three illustrations and two hand-made
spreadsheets. Nothing downloads or regenerates them; they are committed to the repository.

| File | Used for |
| ---- | -------- |
| `04-figures-main/figure_1/assets/Fig1 a (cropped).pdf` | **Figure 1a** — the study-flow strip across the top of Figure 1 |
| `04-figures-main/figure_1/assets/OT_helix_colour_RGB.png` | the Open Targets helix at the centre of the Figure 1d circular Manhattan |
| `05-figures-supplementary/extended_data/assets/extended_figure_1.pdf` | **Extended Data Fig. 1** — the data flowchart |
| `06-supplementary-tables/assets/ST8_-_GSEA_results.xlsx` | **Supplementary Table 8** — GSEA, written by hand |
| `06-supplementary-tables/assets/ST16_-_coloc_overlap.xlsx` | **Supplementary Table 16** — colocalisation overlap, compiled by hand |

Any distribution of this repository must carry them. Without `Fig1 a (cropped).pdf`,
`Figure_1_combined.R` cannot assemble Figure 1; Extended Data Fig. 1, ST8 and ST16 have no other
source.

Extended Data Fig. 1 has roughly twenty pipeline numbers drawn into the artwork. They all match the
current release, but they are not regenerated, so the figure must be redrawn by hand if a
qualifying definition changes.